# Unbiased Feature Importance — Demo

This notebook demonstrates the `unbiased_feature_importance` package, which provides sklearn-compatible
Random Forest classes with unbiased feature importances (arXiv:1903.05179).

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from unbiased_feature_importance import UnbiasedRandomForestClassifier, UnbiasedRandomForestRegressor

## 1. Classification: Unbiased vs Standard Feature Importances

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
feature_names = load_breast_cancer().feature_names

# Fit unbiased classifier
clf = UnbiasedRandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

# Compare with standard MDI importances
std_clf = RandomForestClassifier(n_estimators=100, random_state=42)
std_clf.fit(X, y)

print("Feature importances comparison (top 10):")
print(f"{'Feature':<35} {'Unbiased UFI':>14} {'Standard MDI':>14}")
print("-" * 65)
idx = np.argsort(clf.feature_importances_)[::-1][:10]
for i in idx:
    print(f"{feature_names[i]:<35} {clf.feature_importances_[i]:>14.4f} {std_clf.feature_importances_[i]:>14.4f}")

## 2. SelectFromModel — Automatic Feature Selection

In [ ]:
selector = SelectFromModel(
    UnbiasedRandomForestClassifier(n_estimators=100, random_state=42)
)
selector.fit(X, y)
X_selected = selector.transform(X)

print(f"Original features : {X.shape[1]}")
print(f"Selected features : {X_selected.shape[1]}")
print(f"Selected names    : {feature_names[selector.get_support()].tolist()}")

## 3. Pipeline Integration

In [ ]:
pipe = Pipeline([
    ('select', SelectFromModel(UnbiasedRandomForestClassifier(n_estimators=50, random_state=42))),
    ('clf', UnbiasedRandomForestClassifier(n_estimators=100, random_state=42)),
])
pipe.fit(X, y)
print(f"Pipeline accuracy: {pipe.score(X, y):.4f}")

## 4. Regression Example

In [ ]:
X_r, y_r = load_diabetes(return_X_y=True)
feature_names_r = load_diabetes().feature_names

reg = UnbiasedRandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_r, y_r)

print("Regression feature importances:")
idx = np.argsort(reg.feature_importances_)[::-1]
for i in idx:
    print(f"  {feature_names_r[i]:<6}: {reg.feature_importances_[i]:.4f}")

## 5. sklearn API Compatibility

The classes are full sklearn estimators: `get_params`, `set_params`, `clone`, and cross-validation all work.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

# clone works
clf2 = clone(clf)
print("Cloned estimator params (n_estimators):", clf2.get_params()['n_estimators'])

# cross-validation works
scores = cross_val_score(
    UnbiasedRandomForestClassifier(n_estimators=50, random_state=42),
    X, y, cv=3
)
print(f"CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")